<a href="https://colab.research.google.com/github/jl17243-commits/Statistical-Arbitrage/blob/main/Statistical_Arbitrage_Better_Hedge_Ratio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
import warnings
import yfinance as yf
warnings.filterwarnings('ignore')

In [ ]:
def fetch_price_data(ticker1,ticker2,start,end):
  raw=yf.download([ticker1,ticker2],start=start,end=end,auto_adjust=True,progress=False)
  closes=raw["Close"][[ticker1,ticker2]].dropna()
  return closes

In [ ]:
def ols_hedge_ratio(p,q):
  q_col=q.reshape(-1,1)
  model=LinearRegression().fit(q_col,p)
  return float(model.coef_[0])

def tls_hedge_ratio(p,q):
  X=np.column_stack([p,q])
  X_centered=X-X.mean(axis=0)
  pca=PCA(n_components=2)
  pca.fit(X_centered)
  w_p,w_q=pca.components_[0]
  return float(w_p/w_q)

def compute_hedge_ratios(df):
  t1,t2=df.columns[0],df.columns[1]
  p=df[t1].values
  q=df[t2].values

  ols_fwd=ols_hedge_ratio(p,q)
  tls_fwd=tls_hedge_ratio(p,q)
  ols_rev=ols_hedge_ratio(q,p)
  tls_rev=tls_hedge_ratio(q,p)

  ols_expected_rev=1.0/ols_fwd if ols_fwd!=0 else np.nan
  tls_expected_rev=1.0/tls_fwd if tls_fwd!=0 else np.nan

  ols_inconsistency=abs(ols_rev-ols_expected_rev)
  tls_inconsistency=abs(tls_rev-tls_expected_rev)

  return {
      "pair":f"{t1}/{t2}",
      "ols_fwd":ols_fwd,
      "ols_rev":ols_rev,
      "ols_expected_rev":ols_expected_rev,
      "ols_inconsistency":ols_inconsistency,
      "tls_fwd":tls_fwd,
      "tls_rev":tls_rev,
      "tls_expected_rev":tls_expected_rev,
      "tls_inconsistency":tls_inconsistency
  }

def compute_spreads(df,beta_ols,beta_tls):
  p=df.iloc[:,0]
  q=df.iloc[:,1]
  spreads=pd.DataFrame(index=df.index)
  spreads["ols_spread"]=p-beta_ols*q
  spreads["tls_spread"]=p-beta_tls*q
  return spreads

def print_results(res: dict) -> None:
  """Print hedge ratio comparison results for a single pair."""
  pair = res["pair"]
  sep  = "-" * 52

  print(f"\n{'='*52}")
  print(f"  Pair: {pair}")
  print(f"{'='*52}")

  print(f"\n  {'':20s} {'Fwd (P~Q)':>10s}  {'Rev (Q~P)':>10s}  {'1/Fwd':>10s}")
  print(f"  {sep}")
  print(f"  {'OLS':20s} {res['ols_fwd']:>10.4f}  {res['ols_rev']:>10.4f}  "
        f"{res['ols_expected_rev']:>10.4f}")
  print(f"  {'TLS':20s} {res['tls_fwd']:>10.4f}  {res['tls_rev']:>10.4f}  "
        f"{res['tls_expected_rev']:>10.4f}")

  print(f"\n  Inconsistency (|Rev - 1/Fwd|):")
  print(f"    OLS : {res['ols_inconsistency']:.6f}  {'<-- asymmetric!' if res['ols_inconsistency'] > 1e-6 else 'OK'}")
  print(f"    TLS : {res['tls_inconsistency']:.6f}  {'OK perfectly symmetric' if res['tls_inconsistency'] < 1e-4 else ''}")

  diff_pct = abs(res['tls_fwd'] - res['ols_fwd']) / abs(res['ols_fwd']) * 100
  print(f"\n  TLS vs OLS forward ratio difference: {diff_pct:.1f}%")



In [ ]:
from scipy.sparse import sparray
def generate_signals(spread,z_entry=1.0,z_exit=0.0):
  mu=spread.expanding().mean()
  std=spread.expanding().std()
  z=(spread-mu)/std

  signal=pd.Series(0,index=spread.index,dtype=int)
  position=0

  for i in range(1,len(z)):
    zi=z.iloc[i]
    if np.isnan(zi):
      signal.iloc[i]=0
      continue

    if position==0:
      if zi>z_entry:
        position=-1
      elif zi<-z_entry:
        position=1
    elif position==-1:
      if zi<z_exit:
        position=0
    else:
      if zi>z_exit:
        position=0

    signal.iloc[i]=position

  return signal

def compute_position_sizes(df,beta,signal,capital=100,leverage=2):
  t1,t2=df.columns[0],df.columns[1]
  p_price=df[t1]
  q_price=df[t2]

  max_notioal=leverage*capital
  notional_per_unit=p_price+beta*q_price
  n_p_base=max_notioal/notional_per_unit
  pos_p=signal*n_p_base
  pos_q=-signal*beta*n_p_base

  positions=pd.DataFrame({
      "signal":signal,
      "pos_p":pos_p,
      "pos_q":pos_q
  },index=df.index)

  return positions

def compute_daily_pnl(df,positions):
  t1,t2=df.columns[0],df.columns[1]
  delta_p=df[t1].diff()
  delta_q=df[t2].diff()

  pnl=(positions["pos_p"].shift(1)*delta_p+positions["pos_q"].shift(1)*delta_q)
  return pnl.fillna(0)

def compute_performance_metrics(pnl,capital):
  cumulative=pnl.cumsum()
  total_pnl=cumulative.iloc[-1]

  active_pnl=pnl[pnl!=0]
  sharpe=(active_pnl.mean()/active_pnl.std())*np.sqrt(252)

  running_max=cumulative.cummax()
  drawdown=cumulative-running_max
  max_dd=drawdown.min()
  max_dd_pct=max_dd/capital*100

  return{
      "total_pnl":total_pnl,
      "sharpe":sharpe,
      "max_dd":max_dd,
      "max_dd_pct":max_dd_pct
  }

def run_backtest(df,beta,label,capital=100,leverage=2,z_entry=1,z_exit=0):
  spread=df.iloc[:,0]-beta*df.iloc[:,1]
  signal=generate_signals(spread,z_entry=z_entry,z_exit=z_exit)
  positions=compute_position_sizes(df,beta,signal,capital,leverage)
  pnl=compute_daily_pnl(df,positions)
  metrics=compute_performance_metrics(pnl,capital)

  return {
      "label":label,
      "beta":beta,
      "signal":signal,
      "pnl":pnl,
      "cum_pnl":pnl.cumsum(),
      "spread":spread,
      **metrics,
  }

def print_backtest_results(pair,bt_ols,bt_tls):
  print(f"\n{'='*58}")
  print(f"  Backtest Results — {pair}")
  print(f"{'='*58}")
  print(f"  {'Metric':<22} {'OLS':>14}  {'TLS':>14}")
  print(f"  {'-'*54}")
  print(f"  {'Beta (hedge ratio)':<22} {bt_ols['beta']:>14.4f}  {bt_tls['beta']:>14.4f}")
  print(f"  {'Total PnL ($)':<22} {bt_ols['total_pnl']:>14,.0f}  {bt_tls['total_pnl']:>14,.0f}")
  print(f"  {'Sharpe Ratio':<22} {bt_ols['sharpe']:>14.3f}  {bt_tls['sharpe']:>14.3f}")
  print(f"  {'Max Drawdown ($)':<22} {bt_ols['max_dd']:>14,.0f}  {bt_tls['max_dd']:>14,.0f}")
  print(f"  {'Max Drawdown (%)':<22} {bt_ols['max_dd_pct']:>13.2f}%  {bt_tls['max_dd_pct']:>13.2f}%")
  print(f"  {'Active trading days':<22} {int((bt_ols['signal']!=0).sum()):>14d}  "
    f"{int((bt_tls['signal']!=0).sum()):>14d}")

def plot_backtest(df,bt_ols,bt_tls,pair,axes):
  ax_spread, ax_signal, ax_cum, ax_hist = axes


  for bt,color,ls in [(bt_ols, "#185FA5", "-"),
              (bt_tls, "#1D9E75", "--")]:
    spr=bt["spread"]
    mu=spr.expanding().mean()
    std=spr.expanding().std()
    ax_spread.plot(spr.index, spr, color=color, linewidth=0.8,
                    alpha=0.8, linestyle=ls, label=bt["label"])
    ax_spread.plot(spr.index, mu + std, color=color,
                    linewidth=0.6, linestyle=":", alpha=0.5)
    ax_spread.plot(spr.index, mu - std, color=color,
                    linewidth=0.6, linestyle=":", alpha=0.5)

  ax_spread.axhline(0, color="gray", linewidth=0.5, linestyle="--")
  ax_spread.set_title(f"{pair} — Spread & Entry Bands (±1σ)", fontsize=9, fontweight="bold")
  ax_spread.legend(fontsize=7)
  ax_spread.set_ylabel("Spread", fontsize=8)
  ax_spread.grid(True, alpha=0.25, linewidth=0.4)

  for bt, color, offset in [(bt_ols, "#185FA5", 0.05),
                  (bt_tls, "#1D9E75", -0.05)]:
    sig = bt["signal"]
    ax_signal.step(sig.index, sig + offset, where="post",
                    color=color, linewidth=0.8, alpha=0.8, label=bt["label"])

  ax_signal.set_yticks([-1,0,1])
  ax_signal.set_yticklabels(["Short spread", "Flat", "Long spread"], fontsize=7)
  ax_signal.set_title("Trading Signal", fontsize=9, fontweight="bold")
  ax_signal.legend(fontsize=7)
  ax_signal.grid(True, alpha=0.25, linewidth=0.4)

  ax_cum.plot(bt_ols["cum_pnl"].index, bt_ols["cum_pnl"],
              color="#185FA5", linewidth=1.2, label=f"OLS  Sharpe={bt_ols['sharpe']:.2f}")
  ax_cum.plot(bt_tls["cum_pnl"].index, bt_tls["cum_pnl"],
              color="#1D9E75", linewidth=1.2, label=f"TLS  Sharpe={bt_tls['sharpe']:.2f}")
  ax_cum.axhline(0, color="gray", linewidth=0.5, linestyle="--")
  ax_cum.set_title("Cumulative PnL ($)", fontsize=9, fontweight="bold")
  ax_cum.legend(fontsize=7)
  ax_cum.set_ylabel("PnL ($)", fontsize=8)
  ax_cum.yaxis.set_major_formatter(
      plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
  ax_cum.grid(True, alpha=0.25, linewidth=0.4)

  ols_active = bt_ols["pnl"][bt_ols["pnl"] != 0]
  tls_active = bt_tls["pnl"][bt_tls["pnl"] != 0]

  bins = np.linspace(
      min(ols_active.min(), tls_active.min()),
      max(ols_active.max(), tls_active.max()),
      40
  )
  ax_hist.hist(ols_active, bins=bins, color="#185FA5",
                alpha=0.55, label="OLS", density=True)
  ax_hist.hist(tls_active, bins=bins, color="#1D9E75",
                alpha=0.55, label="TLS", density=True)
  ax_hist.axvline(0, color="gray", linewidth=0.8, linestyle="--")
  ax_hist.set_title("Daily PnL Distribution", fontsize=9, fontweight="bold")
  ax_hist.legend(fontsize=7)
  ax_hist.set_xlabel("Daily PnL ($)", fontsize=8)
  ax_hist.grid(True, alpha=0.25, linewidth=0.4)

  for ax in axes:
    ax.tick_params(axis='x', labelsize=7)
    ax.tick_params(axis='y', labelsize=7)

In [ ]:
PAIRS=[
    ("AAPL","GOOG"),
    ("IBM","SPY"),
    ("DIA","SPY")
]
START="2001-01-01"
END="2023-01-01"

all_results=[]
all_spreads=[]
all_dfs=[]

print("=" * 52)
print(" Better Hedge Ratios: OLS vs TLS ")
print(f" Time Range：{START} → {END}")
print("=" * 52)

for t1, t2 in PAIRS:
  df  = fetch_price_data(t1, t2, start=START, end=END)
  res = compute_hedge_ratios(df)
  spr = compute_spreads(df, beta_ols=res["ols_fwd"], beta_tls=res["tls_fwd"])

  print_results(res)

  all_results.append(res)
  all_spreads.append(spr)
  all_dfs.append(df)

print(" Part 2: Backtest with 2x Leverage Constraint")
print("=" * 70)
CAPITAL=100
LEVERAGE=2
all_bt_ols=[]
all_bt_tls=[]
for df,res in zip(all_dfs, all_results):
  pair=res["pair"]
  bt_ols=run_backtest(df,beta=res["ols_fwd"],label="OLS",capital=CAPITAL,leverage=LEVERAGE)
  bt_tls=run_backtest(df,beta=res["tls_fwd"],label="TLS",capital=CAPITAL,leverage=LEVERAGE)
  print_backtest_results(pair,bt_ols,bt_tls)
  all_bt_ols.append(bt_ols)
  all_bt_tls.append(bt_tls)

fig2,axes2=plt.subplots(3,4,figsize=(18,12))
fig2.suptitle("PnL Backtest — OLS vs TLS (2x Leverage)",
          fontsize=14,fontweight="bold", y=0.99)

for row,(df,res,bt_ols,bt_tls) in enumerate(
        zip(all_dfs,all_results,all_bt_ols,all_bt_tls)):
    plot_backtest(df,bt_ols,bt_tls,res["pair"],axes=axes2[row])

    axes2[row][0].set_ylabel(f"{res['pair']}\nSpread", fontsize=8)

fig2.tight_layout(rect=[0,0,1,0.97])
plt.show()